In [ ]:
from __future__ import print_function
import argparse
import os
import numpy as np
import tensorflow as tf

from model import ConvNet

parser = argparse.ArgumentParser(description='TensorFlow Super Res Example')
parser.add_argument('--model', type=str, required=True, help='model file to use')
parser.add_argument('--path', type=str, default='./test', help='where to run the model(train, test or check)')
parser.add_argument('--cuda', action='store_true', help='use cuda')
opt = parser.parse_args()


def sdmkdir(x):
    if not os.path.isdir(x):
        os.makedirs(x)


def data_process(filepath, outname):
    y = np.load(filepath)
    y = y.astype(np.float32)
    y = y[5::,]
    input_data = np.transpose(y, (1, 2, 0))[np.newaxis, ...]

    model = ConvNet()
    model.build(input_data.shape)
    model.load_weights(opt.model)

    if opt.cuda:
        with tf.device('/GPU:0'):
            out = model(input_data, training=False)
    else:
        out = model(input_data, training=False)

    out_y = out[0].numpy()
    out_y = np.transpose(out_y, (2, 0, 1))
    out_y = np.concatenate((out_y, out_y, out_y), axis=0)

    print(out_y.shape)
    print('output data saved to ', outname)
    np.save(outname, out_y)


def path_process(path, out):
    sdmkdir(out)
    all_files = []
    for x in os.listdir(path):
        if x.endswith(".npy"):
            all_files.append([os.path.join(path, x), x])
    for x, name in all_files:
        data_process(x, os.path.join(out, name))


if __name__ == '__main__':
    input_path = '{}/input/'.format(opt.path)
    output_path = '{}/output'.format(opt.path)
    path_process(input_path, output_path)
